## Controlling Output  

In [1]:
# Setup
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

# Loads variables from the environment
from os import getenv
from dotenv import load_dotenv
load_dotenv()
open_api_key = getenv("ANTHROPIC_API_KEY")

In [2]:
# Helper Functions
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)
    
def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text


##### Message Prefilling example

In [3]:
messages = []

add_user_message(
    messages,
    "Is tea or coffee better at breakfast?"
)

add_assistant_message(
    messages,
    "Coffee  is better because"
)

answer = chat(messages)

answer

' it tends to be more effective at helping people wake up and feel alert in the morning due to its higher caffeine content. Many people also find the ritual of drinking coffee signals the start of their day.\n\nThat said, tea can be a great breakfast choice too - it\'s gentler on the stomach, provides a more gradual energy boost, and green teas offer antioxidants. If you\'re sensitive to caffeine or prefer a calmer morning routine, tea might work better for you.\n\nThe "better" option really depends on your personal preferences, caffeine tolerance, and what you\'re pairing it with. What matters most is choosing something you enjoy that fits your morning routine!'

##### Stop Sequences example

In [4]:
messages = []

add_user_message(messages, "Count from 1 to 10")
answer = chat(messages, stop_sequences=[", 5"])

answer

'1, 2, 3, 4'

##### Structured Data example

In [5]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

text

'\n{\n  "Name": "OrderProcessingRule",\n  "EventPattern": {\n    "source": ["myapp.orders"],\n    "detail-type": ["Order Placed"]\n  },\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n'

In [6]:
import json
json.loads(text.strip())

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}

##### Exercise

- Use massage prefiling and stop sequences only to get the three different commands in a single response
- There shouldn't be any commenmts or explanation 
- Hint: message prefilling isn't limited to just characters like ` ``` `

In [9]:
messages = []

prompt = """
Generate the three different sample AWS CLI commands. Each should be very short. 
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments or explanation:\n```bash")

text = chat(messages, stop_sequences=["```"])
text.strip()

'aws s3 ls s3://my-bucket\naws ec2 describe-instances\naws lambda list-functions'

In [10]:
from IPython.display import Markdown

Markdown(text)


aws s3 ls s3://my-bucket
aws ec2 describe-instances
aws lambda list-functions
